# 01 — What Is a Walk?

Before anything sophisticated, we need to see one walk happen.

## The setup

Imagine a person standing on a number line at position 0. They flip a fair coin:

- **Heads** → step one unit to the right (+1)
- **Tails** → step one unit to the left (−1)

Then they flip again. And again. A thousand times.

Where do they end up? The answer changes every time — that's what *random* means. But the *shape* of how they got there is what we want to look at.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## The math, in one line

A random walk's position after $t$ steps is just the **running total** of every step taken so far:

$$X_t = \sum_{i=1}^{t} \text{step}_i$$

If steps are ±1, the position is the count of "rights" minus the count of "lefts." That's it. No physics, no calculus.

In code, the running total is `np.cumsum`.

In [ ]:
N_STEPS = 1000
SEED = 42  # change this (or set to None) to see a different walk

rng = np.random.default_rng(SEED)
steps = rng.choice([-1, 1], size=N_STEPS)

# Prepend a 0 so the path starts at the origin at step 0.
positions = np.concatenate([[0], np.cumsum(steps)])

print(f"final position after {N_STEPS} steps: {positions[-1]}")
print(f"highest position reached:           {positions.max()}")
print(f"lowest  position reached:           {positions.min()}")

### What just happened

- `rng = np.random.default_rng(SEED)` — a random number generator. Seeding it makes the "randomness" reproducible: same seed → exact same walk.
- `rng.choice([-1, 1], size=1000)` — give me an array of 1000 numbers, each chosen at random from `[-1, 1]`. That's our 1000 coin flips, encoded as ±1.
- `np.cumsum(steps)` — the cumulative sum. After step 1 it's the first step; after step 2 it's the first plus the second; and so on. Exactly the walker's position over time.
- `np.concatenate([[0], ...])` — prepend a 0 so the walk starts at the origin before any flips.

Now let's see it.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(positions, linewidth=1)
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", alpha=0.7)
ax.set_xlabel("step number (t)")
ax.set_ylabel("position")
ax.set_title(f"A single random walk — {N_STEPS} coin-flip steps")
ax.grid(alpha=0.3)
plt.show()

## What to notice

**1. It doesn't drift in any particular direction.** No "up trend" or "down trend" at the level of the underlying coin — every flip is fair. But by chance, this particular walk happens to wander up early, sink to around −20 in the middle, and climb back to the high teens by the end. That entire shape is *noise*, not signal.

**2. It crosses zero only a handful of times** — far fewer than you'd guess. Random walks tend to "stick" on one side of zero for surprisingly long stretches.

**3. It's wiggly at every scale.** Zoom into any 100-step window and you see the same kind of wiggle as the whole 1000-step plot. This *self-similarity* is one of the reasons random walks turn out to be such a deep object.

**4. The position never gets very large compared to the number of steps.** The walker took **1000 steps**, but never went farther than about **±25** from where it started.

That last point is the key. Hold onto it.

## Watch it unfold

Seeing the final picture is one thing. Watching the walk *happen* — left, right, left, left, right — gives you a different feel for it. Run the cell below and use the playback controls to scrub through time.

In [ ]:
STEPS_PER_FRAME = 10  # 1000 steps / 10 = 100 frames
n_frames = N_STEPS // STEPS_PER_FRAME

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.set_xlim(0, N_STEPS)
ax.set_ylim(positions.min() - 2, positions.max() + 2)
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", alpha=0.7)
ax.set_xlabel("step number (t)")
ax.set_ylabel("position")
ax.set_title("Watch the walk unfold")
ax.grid(alpha=0.3)
(line,) = ax.plot([], [], linewidth=1)

def update(frame):
    end = (frame + 1) * STEPS_PER_FRAME + 1  # +1 accounts for the prepended 0
    line.set_data(np.arange(end), positions[:end])
    return (line,)

ani = FuncAnimation(fig, update, frames=n_frames, interval=40, blit=True)
plt.close(fig)  # prevent the static figure from also rendering
HTML(ani.to_jshtml())

## The big hint

Why didn't 1000 steps take the walker farther? Because the +1's and the −1's mostly cancel out. Roughly half the steps push right, roughly half push left, and the leftovers are what determine where the walker ends up.

How big is "the leftovers"? Here's a number to remember:

$$\sqrt{1000} \approx 31.6$$

The walk's final position, its highest point, its lowest point — all comfortably within ±32 of the start. Not ±1000. Not ±100. About √1000.

This is not a coincidence. **The typical distance a random walk travels in $t$ steps grows like $\sqrt{t}$, not $t$.** That's the headline result we'll spend the next few notebooks building intuition for. But you've already seen it once, in this single picture.

## Try this

Edit the parameters above and re-run. The whole notebook is small enough that nothing will break.

- **Change `SEED`** to something else (try `1`, `7`, `99`). Each seed gives a different walk. How different are the final positions? Are any of them close to ±1000?
- **Change `N_STEPS`** to `100`, `10_000`, `1_000_000`. Notice how the *typical magnitude* of the position scales — it grows, but slowly. Compare it to $\sqrt{N}$ each time.
- **Set `SEED = None`** and re-run several times. You're now seeing genuine randomness — different walk every time.
- **Bias the coin** by changing `[-1, 1]` to e.g. `[-1, -1, 1]` (twice as likely to step left). What does the picture look like now? Is it still a "random walk" in the same sense?

Once you've stared at one walk for a while, head to [`../02_many_paths/`](../02_many_paths/) — where we run a thousand walks side-by-side and start to see the *distribution* of where they end up.